In [12]:
from pathlib import Path
from collections import deque
import math
import json

import pandas as pd
import geopandas as gpd
import folium
import h3                           # pip install h3
from shapely import wkt
from shapely.geometry import Polygon, mapping

# --- Config ---
# H3 resolution 8: edge length ~531 m, hex diameter (vertex-to-vertex) ~1062 m ≈ 1 km
H3_INITIAL_RESOLUTION = 8
MAX_H3_RESOLUTION     = 10         # 2 levels deep: res 8 → 9 → 10
SOURCE_CRS = "EPSG:4326"           # H3 operates natively in WGS84

# Official H3 average edge lengths in meters (edge_length == circumradius for H3 hexagons).
H3_EDGE_LENGTH_M = {
    7:  1406.376,
    8:   531.415,
    9:   200.821,
    10:   75.864,
}

INNER_LONDON_BOROUGHS = [
    "Camden",
    "City of London",
    "Hackney",
    "Hammersmith and Fulham",
    "Islington",
    "Kensington and Chelsea",
    "Lambeth",
    "Lewisham",
    "Southwark",
    "Tower Hamlets",
    "Wandsworth",
    "Westminster",
]

In [13]:
def find_boundary_json() -> Path:
    candidates = [
        Path("inner_london_boundary.json"),
        Path("server/scripts/visualize_division/inner_london_boundary.json"),
    ]
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError("inner_london_boundary.json not found.")


def load_inner_london_boundary_from_json(json_path: Path):
    """
    Load preformatted Inner London boundary from GeoJSON.
    Expects a FeatureCollection with one boundary feature.
    """
    gdf = gpd.read_file(json_path)
    if gdf.empty:
        raise ValueError("inner_london_boundary.json has no features.")

    # Ensure WGS84 for H3 operations.
    if gdf.crs is None:
        gdf = gdf.set_crs(SOURCE_CRS)
    elif str(gdf.crs) != SOURCE_CRS:
        gdf = gdf.to_crs(SOURCE_CRS)

    inner_union = gdf.unary_union
    return gdf, inner_union


# --- H3 API compatibility wrappers (handles both h3-py v3 and v4) ---

def _h3_polyfill(poly, resolution: int) -> set:
    """Fill a shapely Polygon (WGS84) with H3 cell IDs."""
    geo = mapping(poly)  # GeoJSON dict: coordinates in [lng, lat] order
    try:
        return set(h3.geo_to_cells(geo, resolution))  # h3-py v4
    except AttributeError:
        return set(h3.polyfill(geo, resolution, geo_json_conformant=True))  # h3-py v3


def _h3_cell_to_shapely(cell_id: str) -> Polygon:
    """Convert an H3 cell to a Shapely Polygon in WGS84."""
    try:
        boundary = h3.cell_to_boundary(cell_id)  # v4 -> [(lat, lng), ...]
    except AttributeError:
        boundary = h3.h3_to_geo_boundary(cell_id)  # v3 -> [(lat, lng), ...]
    return Polygon([(lng, lat) for lat, lng in boundary])  # shapely wants (lng, lat)


def _h3_cell_center(cell_id: str) -> tuple:
    """Return (lat, lng) center of an H3 cell."""
    try:
        return h3.cell_to_latlng(cell_id)  # v4
    except AttributeError:
        return h3.h3_to_geo(cell_id)  # v3


def _h3_get_children(cell_id: str, child_res: int) -> set:
    """Return the ~7 child H3 cells at child_res."""
    try:
        return set(h3.cell_to_children(cell_id, child_res))  # v4
    except AttributeError:
        return set(h3.h3_to_children(cell_id, child_res))  # v3


# --- Tile generation ---

def get_seed_cells(union_wgs84, resolution: int, bbox_buffer_deg: float = 0.03) -> gpd.GeoDataFrame:
    """
    Generate boundary-covering seed H3 cells for Inner London.

    Why not plain polyfill-only:
    - H3 polyfill is center-based, so edge cells whose centers are just outside
      the polygon can be missed.

    Strategy used:
    1) Build candidate cells from a buffered bounding box around the union.
    2) Keep only cells whose hex polygon intersects the union boundary polygon.

    This guarantees boundary-touching coverage while keeping the set finite.
    """
    minx, miny, maxx, maxy = union_wgs84.bounds
    bbox_poly = Polygon(
        [
            (minx - bbox_buffer_deg, miny - bbox_buffer_deg),
            (maxx + bbox_buffer_deg, miny - bbox_buffer_deg),
            (maxx + bbox_buffer_deg, maxy + bbox_buffer_deg),
            (minx - bbox_buffer_deg, maxy + bbox_buffer_deg),
        ]
    )

    candidate_cells = _h3_polyfill(bbox_poly, resolution)

    edge_m = H3_EDGE_LENGTH_M.get(resolution, 0)
    rows = []
    for cell_id in candidate_cells:
        geom = _h3_cell_to_shapely(cell_id)
        if not geom.intersects(union_wgs84):
            continue
        lat, lng = _h3_cell_center(cell_id)
        rows.append(
            {
                "tile_id": cell_id,
                "h3_res": resolution,
                "level": 0,
                "center_lat": lat,
                "center_lon": lng,
                "tile_size_m": edge_m,
                "geometry": geom,
            }
        )

    return gpd.GeoDataFrame(rows, geometry="geometry", crs=SOURCE_CRS)


def subdivide_h3_cell(row: dict) -> list:
    """
    Split one H3 cell into its ~7 children at the next resolution.
    Each hexagon produces exactly 7 children in H3's hierarchy.
    """
    child_res = int(row["h3_res"]) + 1
    child_level = int(row["level"]) + 1
    edge_m = H3_EDGE_LENGTH_M.get(child_res, 0)
    out = []
    for child_id in _h3_get_children(row["tile_id"], child_res):
        lat, lng = _h3_cell_center(child_id)
        out.append(
            {
                "tile_id": child_id,
                "h3_res": child_res,
                "level": child_level,
                "center_lat": lat,
                "center_lon": lng,
                "tile_size_m": edge_m,
                "geometry": _h3_cell_to_shapely(child_id),
            }
        )
    return out


# --- Search radius helper ---

def make_search_radius_layer(cells_gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Build circle geometries representing the Places API search radius per cell.

    For a regular hexagon the circumradius equals the edge length, so a circle
    of radius = edge_length covers all six vertices. A 5% buffer is added to
    ensure complete coverage regardless of H3's slight non-regularity.

    Buffering is done in EPSG:27700 (British National Grid, metres) for accurate
    metric distances, then reprojected back to WGS84 for Folium.
    """
    cols = ["tile_id", "h3_res", "level", "center_lat", "center_lon", "tile_size_m"]
    if "result_count" in cells_gdf.columns:
        cols.append("result_count")
    df = cells_gdf[cols].copy()
    df["search_radius_m"] = (df["tile_size_m"] * 1.05).round(1)

    pts = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["center_lon"], df["center_lat"]),
        crs=SOURCE_CRS,
    )
    pts_bng = pts.to_crs("EPSG:27700")
    pts_bng["geometry"] = pts_bng.apply(
        lambda r: r["geometry"].buffer(r["search_radius_m"]), axis=1
    )
    return pts_bng.to_crs(SOURCE_CRS)

In [14]:
# --- Load preformatted Inner London boundary & generate seed H3 cells ---
boundary_json_path = find_boundary_json()
inner_boundary_gdf, inner_union = load_inner_london_boundary_from_json(boundary_json_path)

seed_cells = get_seed_cells(inner_union, H3_INITIAL_RESOLUTION)

print(f"Boundary JSON  : {boundary_json_path}")
print(f"Boundary features: {len(inner_boundary_gdf)}")
print(
    f"Seed H3 cells  : {len(seed_cells)}  "
    f"(res {H3_INITIAL_RESOLUTION}, edge ~{H3_EDGE_LENGTH_M[H3_INITIAL_RESOLUTION]:.0f} m, "
    f"diameter ~{H3_EDGE_LENGTH_M[H3_INITIAL_RESOLUTION] * 2:.0f} m)"
)
print("Subdivision    : disabled — will be triggered by Google Places API result count")

C:\Users\kylec\AppData\Local\Temp\ipykernel_33212\1340737909.py:27: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  inner_union = gdf.unary_union


Boundary JSON  : inner_london_boundary.json
Boundary features: 1
Seed H3 cells  : 599  (res 8, edge ~531 m, diameter ~1063 m)
Subdivision    : disabled — will be triggered by Google Places API result count


In [15]:
# --- Folium visualization: original seed H3 hex grid ---
inner_union_gdf = gpd.GeoDataFrame(
    [{"name": "Inner London", "geometry": inner_union}],
    geometry="geometry",
    crs=SOURCE_CRS,
)
bounds     = inner_union_gdf.total_bounds
center_lat = (bounds[1] + bounds[3]) / 2
center_lon = (bounds[0] + bounds[2]) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles="OpenStreetMap")

folium.GeoJson(
    data=inner_union_gdf.to_json(),
    name="Inner London boundary",
    style_function=lambda x: {
        "fillColor": "transparent",
        "color": "#111111",
        "weight": 2.5,
        "fillOpacity": 0,
    },
).add_to(m)

folium.GeoJson(
    data=seed_cells.to_json(),
    name=f"Seed H3 cells (res {H3_INITIAL_RESOLUTION}, ~{H3_EDGE_LENGTH_M[H3_INITIAL_RESOLUTION]:.0f} m edge)",
    style_function=lambda x: {
        "fillColor": "#4C78A8",
        "color": "#2A5783",
        "weight": 1,
        "fillOpacity": 0.15,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["tile_id", "h3_res", "level", "tile_size_m"],
        aliases=["H3 Cell", "Resolution", "Depth", "Edge (m)"],
    ),
).add_to(m)

# Add Places API search radius circles for seed cells.
seed_radius = make_search_radius_layer(seed_cells)
folium.GeoJson(
    data=seed_radius.to_json(),
    name="search_radius",
    style_function=lambda x: {
        "fillColor": "#4C78A8",
        "color": "#2A5783",
        "weight": 1.2,
        "fillOpacity": 0.04,
        "dashArray": "6 4",
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["tile_id", "search_radius_m"],
        aliases=["H3 Cell", "Search radius (m)"],
    ),
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

output_path = "h3hex_seed_map.html"
m.save(output_path)
print(f"Saved seed H3 map to: {output_path}")

Saved seed H3 map to: h3hex_seed_map.html


In [16]:
# --- Mock Google Places API subdivision (H3 hex edition) ---
# Rules:
#   - 50% chance a tile returns 20 results (saturated) → subdivide into ~7 children
#   - 50% chance returns under 20 → accept tile as final
#   - Children fully outside Inner London boundary are discarded immediately

import random

SATURATION_THRESHOLD = 20   # Google Places API result cap per query
MAX_DEPTH            = 4    # res 8 → 9 → 10 → 11 → 12


def mock_places_api(tile_row: dict) -> int:
    """50% chance of saturation (20), else uniform random 0–19."""
    if random.random() < 0.5:
        return SATURATION_THRESHOLD
    return random.randint(0, SATURATION_THRESHOLD - 1)


def run_mock_h3_subdivision(
    seed_cells: gpd.GeoDataFrame,
    union_wgs84,
    max_depth: int = MAX_DEPTH,
    rng_seed: int  = 42,
) -> gpd.GeoDataFrame:
    """
    Simulate adaptive H3 subdivision driven by mock Places API results.

    Each saturated tile is split into ~7 children (H3 hierarchy).
    Children that don't intersect Inner London are dropped without an API call.
    """
    random.seed(rng_seed)
    queue       = deque(seed_cells.to_dict("records"))
    final_cells = []
    stats       = {"api_calls": 0, "subdivided": 0, "discarded_oob": 0}

    while queue:
        row  = queue.popleft()
        geom = row["geometry"]

        # Boundary check — no API call for out-of-bounds cells.
        if not geom.intersects(union_wgs84):
            stats["discarded_oob"] += 1
            continue

        result_count = mock_places_api(row)
        stats["api_calls"] += 1
        row["result_count"] = result_count

        if result_count >= SATURATION_THRESHOLD and row["level"] < max_depth:
            stats["subdivided"] += 1
            for child in subdivide_h3_cell(row):   # ~7 children per hex
                queue.append(child)
        else:
            final_cells.append(row)

    gdf = gpd.GeoDataFrame(final_cells, geometry="geometry", crs=SOURCE_CRS)
    print("Mock run complete")
    print(f"  API calls made  : {stats['api_calls']}")
    print(f"  Cells subdivided: {stats['subdivided']}  (each → ~7 children)")
    print(f"  Discarded (OOB) : {stats['discarded_oob']}")
    print(f"  Final cell count: {len(gdf)}")
    depth_summary = {
        f"L{d} res {H3_INITIAL_RESOLUTION + d} (~{H3_EDGE_LENGTH_M.get(H3_INITIAL_RESOLUTION + d, 0):.0f}m)":
        int((gdf["level"] == d).sum())
        for d in sorted(gdf["level"].unique())
    }
    print(f"  Depth breakdown : {depth_summary}")
    return gdf


mock_cells = run_mock_h3_subdivision(seed_cells, inner_union)

Mock run complete
  API calls made  : 118348
  Cells subdivided: 16960  (each → ~7 children)
  Discarded (OOB) : 971
  Final cell count: 101388
  Depth breakdown : {'L0 res 8 (~531m)': 286, 'L1 res 9 (~201m)': 1018, 'L2 res 10 (~76m)': 3610, 'L3 res 11 (~0m)': 12024, 'L4 res 12 (~0m)': 84450}


In [17]:
# --- Folium visualization: mock adaptive H3 subdivision result ---
LEVEL_COLORS = {
    0: ("#4C78A8", "#2A5783"),   # blue   – res 8,  ~531 m
    1: ("#F58518", "#B05B08"),   # orange – res 9,  ~201 m
    2: ("#54A24B", "#2D6E28"),   # green  – res 10,  ~76 m
    3: ("#E45756", "#9C1C1B"),   # red    – res 11,  ~29 m
    4: ("#B279A2", "#7B3D73"),   # purple – res 12,  ~11 m
}

m2 = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles="OpenStreetMap")

# Inner London boundary
folium.GeoJson(
    data=inner_union_gdf.to_json(),
    name="Inner London boundary",
    style_function=lambda x: {
        "fillColor": "transparent",
        "color": "#111111",
        "weight": 2.5,
        "fillOpacity": 0,
    },
).add_to(m2)

# One toggleable layer per depth level
for level, (fill, stroke) in LEVEL_COLORS.items():
    level_cells = mock_cells[mock_cells["level"] == level]
    if level_cells.empty:
        continue
    res    = H3_INITIAL_RESOLUTION + level
    edge_m = H3_EDGE_LENGTH_M.get(res, 0)

    # Hex polygons
    folium.GeoJson(
        data=level_cells.to_json(),
        name=f"Level {level} - res {res}, ~{edge_m:.0f} m  ({len(level_cells)} cells)",
        style_function=lambda x, f=fill, s=stroke: {
            "fillColor": f,
            "color":     s,
            "weight":    0.8,
            "fillOpacity": 0.25,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["tile_id", "h3_res", "level", "tile_size_m", "result_count"],
            aliases=["H3 Cell", "Resolution", "Depth", "Edge (m)", "Mock results"],
        ),
    ).add_to(m2)

    # Search radius circles
    level_radius = make_search_radius_layer(level_cells)
    folium.GeoJson(
        data=level_radius.to_json(),
        name=f"search_radius L{level} ({len(level_radius)} circles)",
        style_function=lambda x, f=fill, s=stroke: {
            "fillColor": f,
            "color": s,
            "weight": 1.2,
            "fillOpacity": 0.03,
            "dashArray": "6 4",
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["tile_id", "search_radius_m"],
            aliases=["H3 Cell", "Search radius (m)"],
        ),
    ).add_to(m2)

folium.LayerControl(collapsed=False).add_to(m2)

output_path2 = "h3hex_mock_map.html"
m2.save(output_path2)
print(f"Saved mock adaptive H3 map to: {output_path2}")

Saved mock adaptive H3 map to: h3hex_mock_map.html
